# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [ ]:
import os, zipfile, json, sqlite3, time, subprocess, sys
from pathlib import Path
import pandas as pd, requests

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = Path('/content/flasheats_classroom')
if not BASE.exists() or not (BASE / 'database' / 'flasheats.db').exists():
    from google.colab import files
    print('Upload FlashEats_Classroom_Pack.zip')
    uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith('.zip'))
    with zipfile.ZipFile(zip_name) as z: z.extractall('/content')
    extracted=Path('/content/FlashEats_Classroom_Pack')
    if extracted.exists(): extracted.rename(BASE)
print(BASE, BASE.exists())

## Challenge 1 — How large is the late-delivery problem?

**Definition (stated first):** late = `actual_delivery_at > promised_eta` (datetimes, `format='mixed'`); denominator = deduped (`order_id`) delivered orders with non-null actual + promised (`n=1495`); cancelled (68) excluded; 37 delivered-but-unknown excluded from rate and reported separately.

**Result:** 843/1495 ≈ **56.4% late** (>10 min: ~23.3%); median lateness among late ≈ **8.0 min**. Raw table has 1603 rows / 1600 unique orders (3 duplicate `order_id`s); case variant `'Delivered'` (5) counted as delivered.

In [ ]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

In [ ]:
# Challenge 1 — late size on business grain (one row per order_id; datetimes parsed before comparison).
query = '''
SELECT order_id, promised_eta, actual_delivery_at, final_status, traffic_bucket, weather_bucket
FROM orders LIMIT 5;
'''
display(pd.read_sql(query, con))
orders_all = pd.read_sql("SELECT * FROM orders", con)
print("Raw:", len(orders_all), "rows |", orders_all["order_id"].nunique(), "unique orders")
g = orders_all.drop_duplicates("order_id", keep="first").copy()
for c in ["promised_eta", "actual_delivery_at"]:
    g[c + "_p"] = pd.to_datetime(g[c], format="mixed", errors="coerce")
g["status_norm"] = g["final_status"].str.strip().str.lower()
valid = g[(g["status_norm"] == "delivered") & g["actual_delivery_at_p"].notna()].copy()
valid["delay_min"] = (valid["actual_delivery_at_p"] - valid["promised_eta_p"]).dt.total_seconds() / 60
print(f"Valid delivered: {len(valid)} | Late: {int((valid['delay_min'] > 0).sum())} ({(valid['delay_min'] > 0).mean()*100:.2f}%) | median late: {valid.loc[valid['delay_min']>0,'delay_min'].median():.2f} min")
display(valid.sort_values("delay_min", ascending=False).head(5)[["order_id", "promised_eta", "actual_delivery_at", "delay_min"]])

### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause

**Verdict:** associated, not proven. Severe/high traffic (late rate ~0.66) and rain/heavy_rain (~0.67–0.74) show higher median delay than low/clear, but restaurant prep, reassignment and ETA bias are not ruled out. Treat as descriptive, not causal.

In [ ]:
# Challenge 2 — traffic/weather/distance cuts (association only).
df = pd.read_sql("SELECT * FROM orders", con).drop_duplicates("order_id", keep="first").copy()
for c in ["created_at", "promised_eta", "actual_delivery_at"]:
    df[c + "_p"] = pd.to_datetime(df[c], format="mixed", errors="coerce")
df = df[df["actual_delivery_at_p"].notna()].copy()
df["delay_min"] = (df["actual_delivery_at_p"] - df["promised_eta_p"]).dt.total_seconds() / 60
df["traffic_norm"] = df["traffic_bucket"].str.strip().str.lower()
print("Median delay + late rate by traffic:")
display(df.groupby("traffic_norm").agg(median_delay=("delay_min", "median"), late_rate=("delay_min", lambda x: (x > 0).mean()), n=("delay_min", "size")).sort_values("late_rate"))
print("By weather:")
display(df.groupby("weather_bucket").agg(median_delay=("delay_min", "median"), late_rate=("delay_min", lambda x: (x > 0).mean()), n=("delay_min", "size")).sort_values("late_rate"))
print("Hypothesis: severe/high + rain ride with higher delay. Alternative NOT ruled out: peak-hour/restaurant/driver confounders; association != causation.")

## Challenge 3 — Customer Support disagrees

**Verdict:** customers confirm real lateness (late_delivery 38, eta_changed 37, restaurant_delay 37; ticketed valid orders median ~14 min) and add what timestamps miss: ETA churn + ready-but-no-rider + app-vs-reality mismatch. Records NOT unique: 1 duplicate `ticket_id` (T00013), 3 tickets lack `order_id`, 3 orders repeat — dedup before counting.

In [ ]:
tickets = pd.read_csv(BASE / 'data' / 'support_tickets.csv')
display(tickets.head())
print(tickets["category"].value_counts(dropna=False))
print("dup ticket_id:", int(tickets["ticket_id"].duplicated().sum()), "| null order_id:", int(tickets["order_id"].isna().sum()))
if tickets["ticket_id"].duplicated().any():
    display(tickets[tickets.duplicated("ticket_id", keep=False)].sort_values("ticket_id"))

## Challenge 4 — Retrieve Dispatch data reliably

**Verdict:** 1600/1600 records verified across 32 pages (page_size=50) with retry on 500/429; raw pages in `student_output/raw_dispatch/page_*.json`; 95 reassignments; `completed` 1532 / `cancelled` 68; eta-v3.1/v3.2 split 800/800. One 200 never proves completeness — only count-verification does.

In [ ]:
!pip -q install flask
api_proc=subprocess.Popen([sys.executable,str(BASE/'api'/'mock_dispatch_api.py')],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
requests.get('http://127.0.0.1:8000/health').json()

In [ ]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [ ]:
RAW_DIR = BASE / 'student_output' / 'raw_dispatch'
RAW_DIR.mkdir(parents=True, exist_ok=True)
API = 'http://127.0.0.1:8000/dispatch/orders'

def fetch_all_dispatch_orders(page_size=50, max_retries=5, timeout=10):
    records, page, expected = [], 1, None
    while True:
        attempt = 0
        while True:
            r = requests.get(API, params={'page': page, 'page_size': page_size}, timeout=timeout)
            if r.status_code in (429, 500):
                attempt += 1
                if attempt > max_retries:
                    raise RuntimeError(f'Page {page} failing ({r.status_code}) after retries — INCOMPLETE.')
                wait = 1
                try:
                    wait = int(r.json().get('retry_after_seconds', 1))
                except Exception:
                    pass
                time.sleep(wait)
                continue
            r.raise_for_status()
            break
        payload = r.json()
        (RAW_DIR / f'page_{page:03d}.json').write_text(json.dumps(payload, indent=2))
        expected = payload.get('total_records') if expected is None else expected
        records.extend(payload.get('data', []))
        if not payload.get('has_more'):
            break
        page += 1
    if expected is not None and len(records) != expected:
        raise RuntimeError(f'INCOMPLETE: {len(records)} vs server {expected}')
    print(f'Verified {len(records)}/{expected} across {page} pages.')
    return records

dispatch_records = fetch_all_dispatch_orders()
print('Retrieved:', len(dispatch_records))
dd = pd.DataFrame(dispatch_records)
display(dd["dispatch_status"].value_counts(dropna=False))
print('Reassignments:', int((dd["driver_id"] != dd["original_driver_id"]).sum()))

## Challenge 5 — Driver events

**Verdict:** NO reliable arrival-at-restaurant event. Observed: `assigned` (1600), `picked_up` (1532), `delivered` (1532). `gps_ping` (5371) is only an inference signal — sparse, no geofence, 14 picked_up-before-assigned ordering faults. Arrival must be instrumented, not inferred.

In [ ]:
with open(BASE / 'data' / 'driver_events.json') as f:
    driver_events = json.load(f)
display(driver_events[0])
flat = [{'driver_id': d['driver_id'], **e} for d in driver_events for e in d['events']]
dev = pd.DataFrame(flat)
print(dev["type"].value_counts())
print('arrived_at_restaurant events:', int((dev["type"].str.lower() == 'driver_arrived_at_restaurant').sum()))

## Final FDE recommendation

1. **Size:** 843/1495 ≈ 56.4% late (>10 min: 23.3%; median late 8.0 min); 37 unknown + 68 cancelled excluded.
2. **Evidence:** severe/high traffic + rain ride with delay; ticketed orders skew late; 95 dispatch reassignments.
3. **Trusted sources:** orders timestamps/outcomes for KPI; Dispatch API for assignment; tickets for 'why'; status/GPS contextual only.
4. **Uncertainty:** no causal proof on traffic; no restaurant-vs-road split (no arrival event); per-team 56% unstable until grain/casing/null rules fixed.
5. **Next instrumentation:** arrival + handoff events, single owned late definition/taxonomy, dedup + required actuals.
6. **Decision:** NO AI predictor yet — lock definitions/completeness first, else the model learns label noise.
